<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/data_preprocess_v3_macro_ranked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing — v3 (Macro Quantile Mapping)

**Based on:** v2 corrected + macro time-series quantile mapping

Pipeline:
1. Load 94 stock-level characteristics from `datashare.csv`
2. Query WRDS/CRSP for monthly returns + FF risk-free rate
3. Create `exret_lead1` (next month excess return)
4. Load 8 macro variables (Welch & Goyal 2008), lagged by 1 month
5. Cross-sectionally rank ALL 94 characteristics → map to [-1, 1]
6. **NEW: Time-series expanding-window quantile map 8 macro variables → [-1, 1]**
7. Filter to 1957-03 ~ 2016-12, merge macro, save as Parquet

**v3 change vs. v2:**
- Macro variables mapped to [-1, 1] via expanding-window empirical CDF (no look-ahead bias)
- This bounds the macro values during crisis periods (e.g., 2008 `svar`, `dfy`) and prevents
  extreme interaction terms from dominating the ENet objective

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q wrds gspread google-api-python-client pyarrow

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.4 MB/s eta 0:00:00


In [2]:
import wrds
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

All imports OK


## 1. Load datashare.csv (94 Stock-Level Characteristics)

In [3]:
csv_path = "/content/drive/MyDrive/datashare.csv"
df = pd.read_csv(csv_path)
df['DATE'] = pd.to_datetime(df['DATE'], format='%Y%m%d')

print(f"datashare shape: {df.shape}")
print(f"Date range:      {df['DATE'].min().date()} to {df['DATE'].max().date()}")
print(f"Total cols - 2 (permno/DATE) = {df.shape[1]-2} (should be 95 = 94 chars + sic2)")

datashare shape: (4117300, 97)
Date range:      1957-01-31 to 2021-12-31
Total cols - 2 (permno/DATE) = 95 (should be 95 = 94 chars + sic2)


## 2. Query WRDS/CRSP

In [4]:
db = wrds.Connection()

Enter your WRDS username [root]:zixian_zhou
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: n
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [5]:
start_date = df['DATE'].min()
end_date   = df['DATE'].max()

query = f"""
SELECT
    b.permno,
    b.date,
    b.ret,
    c.rf,
    (b.ret - c.rf) AS exret
FROM crsp.msf b
LEFT JOIN ff.factors_monthly c ON
    EXTRACT(YEAR  FROM b.date) = EXTRACT(YEAR  FROM c.date)
    AND EXTRACT(MONTH FROM b.date) = EXTRACT(MONTH FROM c.date)
WHERE b.date >= '{start_date}' AND b.date <= '{end_date}'
  AND b.ret IS NOT NULL
"""

df_msf = db.raw_sql(query)
df_msf['date'] = pd.to_datetime(df_msf['date'])
print(f"CRSP rows: {len(df_msf):,}")

CRSP rows: 4,318,501


In [6]:
df_exret = pd.merge(
    df, df_msf,
    left_on=['permno', 'DATE'],
    right_on=['permno', 'date'],
    how='inner'
).drop(columns=['date'])

print(f"After merge: {len(df_exret):,} rows")

After merge: 4,096,791 rows


## 3. Basic Filters

In [7]:
n0 = len(df_exret)
df_exret = df_exret[df_exret['ret'].notna()].copy()
df_exret = df_exret[df_exret['mvel1'].notna()].copy()
df_exret.reset_index(drop=True, inplace=True)

print(f"Dropped (ret/mvel1 NaN): {n0 - len(df_exret):,}  |  Remaining: {len(df_exret):,}")

Dropped (ret/mvel1 NaN): 2,982  |  Remaining: 4,093,809


## 4. Create Target Variable `exret_lead1`

In [8]:
df_exret = df_exret.sort_values(['permno', 'DATE']).reset_index(drop=True)
df_exret['exret_lead1'] = df_exret.groupby('permno')['exret'].shift(-1)

print(f"exret_lead1 NaN: {df_exret['exret_lead1'].isna().sum():,}  (last obs of each permno)")

exret_lead1 NaN: 32,750  (last obs of each permno)


## 5. Filter to Paper Sample Window: 1957-03 ~ 2016-12

In [9]:
n0 = len(df_exret)

df_exret = df_exret[
    (df_exret['DATE'] >= pd.Timestamp('1957-03-31')) &
    (df_exret['DATE'] <= pd.Timestamp('2016-12-31'))
].copy()
df_exret = df_exret.dropna(subset=['exret_lead1']).reset_index(drop=True)

print(f"Dropped (date/target filter): {n0 - len(df_exret):,}")
print(f"Final rows: {len(df_exret):,}")
print(f"Date range: {df_exret['DATE'].min().date()} to {df_exret['DATE'].max().date()}")

Dropped (date/target filter): 381,001
Final rows: 3,712,808
Date range: 1957-04-30 to 2016-12-30


## 6. Cross-Sectional Rank Normalization (94 Characteristics Only)

Same as v2 — rank all 94 stock characteristics cross-sectionally each month, map to [-1, 1].

In [10]:
NON_CHAR_COLS = {'permno', 'DATE', 'ret', 'rf', 'exret', 'exret_lead1', 'sic2'}
CHAR_COLS_94  = [c for c in df_exret.columns if c not in NON_CHAR_COLS]

print(f"Characteristic columns to rank-normalize: {len(CHAR_COLS_94)}  (should be 94)")
print(f"sic2 in CHAR_COLS_94: {'sic2' in CHAR_COLS_94}  (should be False)")

Characteristic columns to rank-normalize: 94  (should be 94)
sic2 in CHAR_COLS_94: False  (should be False)


In [11]:
def rank_norm_median(s: pd.Series) -> pd.Series:
    """
    Cross-sectional rank normalization (per month):
    1) Fill NaN with cross-sectional median (paper fn.30)
    2) Rank with average ties
    3) Map to [-1, 1]:  mapped = (rank / (N+1)) * 2 - 1
    """
    filled = s.fillna(s.median(skipna=True)).fillna(0.0)
    ranks  = filled.rank(method='average')
    n      = ranks.count()
    return (ranks / (n + 1)) * 2 - 1 if n > 0 else filled


print("Rank-normalizing 94 characteristics cross-sectionally by DATE...")
df_exret[CHAR_COLS_94] = (
    df_exret.groupby('DATE')[CHAR_COLS_94]
    .transform(rank_norm_median)
)
print("Done!")

for c in ['mvel1', 'bm', 'mom12m']:
    print(f"  {c:8s}  min={df_exret[c].min():.4f}  max={df_exret[c].max():.4f}  mean={df_exret[c].mean():+.2e}")

Rank-normalizing 94 characteristics cross-sectionally by DATE...
Done!
  mvel1     min=-0.9998  max=0.9998  mean=+2.15e-17
  bm        min=-0.9990  max=0.9990  mean=+2.66e-18
  mom12m    min=-0.9955  max=0.9956  mean=+2.27e-18


## 7. Load 8 Macroeconomic Predictors (Welch & Goyal 2008)

Variables: `tbl`, `d/p`, `e/p`, `b/m`, `tms`, `dfy`, `ntis`, `svar`

Lagged by 1 month to avoid look-ahead bias.

In [12]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from googleapiclient.discovery import build

creds, _ = default()
gc       = gspread.authorize(creds)
sh       = gc.open("Data2024")

drive_svc   = build('drive', 'v3', credentials=creds)
export_path = "/content/Data2024_export.xlsx"

req = drive_svc.files().export_media(
    fileId=sh.id,
    mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
)
with open(export_path, "wb") as f:
    f.write(req.execute())

df_macro = pd.read_excel(export_path, sheet_name="Monthly")
print(f"Macro sheet shape: {df_macro.shape}")

Macro sheet shape: (1848, 57)


In [13]:
MACRO_COLS = ['tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar']

df_macro = df_macro.sort_values('yyyymm').reset_index(drop=True)
df_macro[MACRO_COLS] = df_macro[MACRO_COLS].shift(1)   # lag by 1 month

df_macro['year_month'] = (
    pd.to_datetime(df_macro['yyyymm'], format='%Y%m').dt.to_period('M')
)
df_macro = df_macro[['yyyymm'] + MACRO_COLS + ['year_month']]

print(f"Macro rows after lag: {len(df_macro)}")

Macro rows after lag: 1848


## 7b. NEW — Time-Series Expanding-Window Quantile Mapping for Macro Variables

**Why:** The 8 macro variables are NOT cross-sectional (same value for all stocks in a month),
so we cannot rank them cross-sectionally. Instead, we use an **expanding-window** approach:

For each month $t$ and each macro variable $z$:
$$z_{t}^{\text{mapped}} = 2 \times \frac{\text{rank of } z_t \text{ among } \{z_1, z_2, \ldots, z_t\}}{t} - 1$$

This:
- Maps each macro variable to [-1, 1] using only **past and current** data (no look-ahead)
- Bounds extreme crisis-period values (e.g., `svar` in 2008 just maps to ~+1.0)
- Eliminates the need for lambda floors in the ENet model

**Minimum window:** We require at least 24 months of history before mapping starts.
Earlier months use a simple min-max rescale to [-1, 1] as fallback.

In [14]:
def expanding_quantile_map(series, min_window=24):
    """
    Map a time-series to [-1, 1] using expanding-window empirical CDF.

    For each time t, compute the rank of x_t among x_1, ..., x_t,
    then map:  mapped = 2 * rank / (t+1) - 1

    No look-ahead bias: only uses data up to and including time t.
    """
    values = series.values.astype(np.float64)
    n = len(values)
    mapped = np.full(n, np.nan)

    for t in range(n):
        if np.isnan(values[t]):
            continue
        # expanding window: all non-NaN values up to and including t
        window = values[:t+1]
        valid_mask = ~np.isnan(window)
        valid_vals = window[valid_mask]
        count = len(valid_vals)

        if count < 2:
            mapped[t] = 0.0  # not enough history, map to center
            continue

        # rank of current value among the expanding window
        rank = np.sum(valid_vals <= values[t])
        # map to [-1, 1]
        mapped[t] = 2.0 * rank / (count + 1) - 1.0

    return pd.Series(mapped, index=series.index)


print("Applying expanding-window quantile mapping to macro variables...")
print()

# Show before vs after for each macro variable
for col in MACRO_COLS:
    raw_min = df_macro[col].min()
    raw_max = df_macro[col].max()
    raw_std = df_macro[col].std()

    df_macro[col] = expanding_quantile_map(df_macro[col])

    new_min = df_macro[col].min()
    new_max = df_macro[col].max()
    new_std = df_macro[col].std()

    print(f"  {col:5s}  raw=[{raw_min:+.4f}, {raw_max:+.4f}] std={raw_std:.4f}"
          f"  →  mapped=[{new_min:+.4f}, {new_max:+.4f}] std={new_std:.4f}")

print()
print("Done! All macro variables now bounded in [-1, 1].")

Applying expanding-window quantile mapping to macro variables...

  tbl    raw=[+0.0001, +0.1630] std=0.0296  →  mapped=[-0.9891, +0.9973] std=0.6452
  d/p    raw=[+0.0108, +0.1536] std=0.0177  →  mapped=[-0.9987, +0.9973] std=0.6241
  e/p    raw=[+0.0079, +0.1882] std=0.0274  →  mapped=[-0.9988, +0.9965] std=0.6574
  b/m    raw=[+0.1205, +2.0285] std=0.2632  →  mapped=[-0.9979, +0.9854] std=0.6405
  tms    raw=[-0.0365, +0.0455] std=0.0132  →  mapped=[-0.9973, +0.9977] std=0.6203
  dfy    raw=[+0.0032, +0.0564] std=0.0068  →  mapped=[-0.9965, +0.9877] std=0.5215
  ntis   raw=[-0.0560, +0.1770] std=0.0258  →  mapped=[-0.9980, +0.9459] std=0.5374
  svar   raw=[+0.0000, +0.0732] std=0.0051  →  mapped=[-0.9944, +0.9988] std=0.5981

Done! All macro variables now bounded in [-1, 1].


## 8. Merge Stock Data with Macro

In [15]:
df_exret['year_month'] = df_exret['DATE'].dt.to_period('M')

df_merged = pd.merge(df_exret, df_macro, on='year_month', how='left')
df_merged = df_merged.drop(columns=['yyyymm', 'year_month'])

print(f"Merged shape: {df_merged.shape}")
print(f"Macro NaN (within 1957-2016, expect 0):")
print(df_merged[MACRO_COLS].isna().sum().to_string())

Merged shape: (3712808, 109)
Macro NaN (within 1957-2016, expect 0):
tbl     0
d/p     0
e/p     0
b/m     0
tms     0
dfy     0
ntis    0
svar    0


## 8b. Verify macro values are bounded

In [16]:
print("Macro variable ranges in merged data:")
for col in MACRO_COLS:
    print(f"  {col:5s}  [{df_merged[col].min():+.4f}, {df_merged[col].max():+.4f}]")

print()
print("Characteristic variable ranges (should also be in [-1, 1]):")
for c in ['mvel1', 'bm', 'mom12m']:
    print(f"  {c:8s}  [{df_merged[c].min():.4f}, {df_merged[c].max():.4f}]")

Macro variable ranges in merged data:
  tbl    [-0.9891, +0.9973]
  d/p    [-0.9987, +0.6851]
  e/p    [-0.9988, +0.9527]
  b/m    [-0.9979, +0.9675]
  tms    [-0.9973, +0.9977]
  dfy    [-0.9965, +0.9648]
  ntis   [-0.9980, +0.8486]
  svar   [-0.9895, +0.9984]

Characteristic variable ranges (should also be in [-1, 1]):
  mvel1     [-0.9998, 0.9998]
  bm        [-0.9990, 0.9990]
  mom12m    [-0.9955, 0.9956]


## 9. Forward-fill sic2 (Industry Code)

In [17]:
if 'sic2' in df_merged.columns:
    n_nan = df_merged['sic2'].isna().sum()
    df_merged['sic2'] = df_merged.groupby('permno')['sic2'].ffill().bfill()
    print(f"sic2 NaN: {n_nan:,} → {df_merged['sic2'].isna().sum():,}")
else:
    print("Warning: sic2 not found")

sic2 NaN: 245,649 → 0


## 10. Final Summary

In [18]:
print("=" * 65)
print("FINAL PREPROCESSED DATA (v3 — macro quantile mapped)")
print("=" * 65)
print(f"Shape:          {df_merged.shape}")
print(f"Date range:     {df_merged['DATE'].min().date()} → {df_merged['DATE'].max().date()}")
print(f"Unique permnos: {df_merged['permno'].nunique():,}")
print(f"Unique months:  {df_merged['DATE'].nunique()}")
print()
print(f"exret_lead1  NaN: {df_merged['exret_lead1'].isna().sum()}")
print(f"             mean={df_merged['exret_lead1'].mean():.6f}  std={df_merged['exret_lead1'].std():.6f}")
print()
print("Rank-normalized chars (values in [-1, 1]):")
for c in ['mvel1', 'bm', 'mom12m']:
    print(f"  {c:10s}  [{df_merged[c].min():.4f}, {df_merged[c].max():.4f}]")
print()
print("Quantile-mapped macro (values in [-1, 1]):")
for c in MACRO_COLS:
    print(f"  {c:5s}  [{df_merged[c].min():+.4f}, {df_merged[c].max():+.4f}]")
print()
print("sic2 sample (raw, NOT normalized):")
print(df_merged['sic2'].value_counts().head(5).to_string())

FINAL PREPROCESSED DATA (v3 — macro quantile mapped)
Shape:          (3712808, 109)
Date range:     1957-04-30 → 2016-12-30
Unique permnos: 29,825
Unique months:  717

exret_lead1  NaN: 0
             mean=0.007313  std=0.172393

Rank-normalized chars (values in [-1, 1]):
  mvel1       [-0.9998, 0.9998]
  bm          [-0.9990, 0.9990]
  mom12m      [-0.9955, 0.9956]

Quantile-mapped macro (values in [-1, 1]):
  tbl    [-0.9891, +0.9973]
  d/p    [-0.9987, +0.6851]
  e/p    [-0.9988, +0.9527]
  b/m    [-0.9979, +0.9675]
  tms    [-0.9973, +0.9977]
  dfy    [-0.9965, +0.9648]
  ntis   [-0.9980, +0.8486]
  svar   [-0.9895, +0.9984]

sic2 sample (raw, NOT normalized):
sic2
67.0    353505
60.0    302308
73.0    264641
28.0    230205
36.0    225301


## 11. Save to Google Drive as Parquet

In [19]:
db.close()
print("WRDS connection closed.")

save_dir = '/content/drive/MyDrive/industry_project'
os.makedirs(save_dir, exist_ok=True)

parquet_path = os.path.join(save_dir, 'preprocess_data_v3_macro_ranked.parquet')
df_merged.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f"Saved Parquet: {parquet_path}")
print(f"File size:     {os.path.getsize(parquet_path) / 1e9:.2f} GB")

WRDS connection closed.
Saved Parquet: /content/drive/MyDrive/industry_project/preprocess_data_v3_macro_ranked.parquet
File size:     1.93 GB
